# coerce-float-arg-to-array — ex3: coerce_kwargs: promote float kwargs to tensors, leave control flags raw

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `coerce-float-arg-to-array`. Running the final beacon cell reports progress against the `Backprop: Coerce float arg to array` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Coerce float arg to array` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`coerce-float-arg-to-array`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "coerce-float-arg-to-array"
DD_SUBTOPIC = "Backprop: Coerce float arg to array"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Coerce **kwargs** values — same scalar rule, different container

Ex1 coerced a single positional arg. Ex2 coerced the args tuple. The third facet: the SAME promote-scalar-to-tensor rule applied across the kwargs dict. PyTorch ops commonly take numeric hyperparameters as kwargs — `t.clamp(x, min=0.0, max=1.0)`, `t.pow(x, exponent=2.0)` — and the wrapper must coerce these the same way it coerces positional scalars so the Recipe stores tensors everywhere.

```python
coerce_kwargs({'min': 0.0, 'max': 1.0, 'keepdim': True, 'dim': 2})
  → {'min': tensor(0.0), 'max': tensor(1.0), 'keepdim': True, 'dim': 2}
```

**Control-flag kwargs stay raw.** `keepdim`, `dim`, `out` are NOT numeric scalars in the math sense — they're flags / axis indices. The raw torch op interprets `dim` as a Python int specifically; passing `tensor(2)` raises. We get this for free by reusing the int/float-only rule from ex1 — but the trap is `dim`: a Python int. The simplest out: coerce ONLY `float` values in kwargs, leave `int` / `bool` alone.

**Why this matters.** Without kwarg coercion, the Recipe stores `kwargs={'min': 0.0}` — a Python float. The reverse pass for clamp needs `min` as a tensor (to compare against `x.array` with masking). Heterogeneous storage forces N tensor-vs-float checks across every back fn — same complaint we had against not coercing positional args.

### Exercise 3 — coerce_kwargs: promote float kwargs to tensors, leave control flags raw

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the float-only coercion rule across a kwargs dict: promote any float-valued entry to a 0-D tensor while leaving int / bool control flags raw so axis-indices and keepdim flags continue to be Python primitives the raw torch fn can use.
> Keywords: kwargs, coerce, float-only, control-flag, wrap-forward
> ```

**KCs targeted:** `kwargs-coerce-float-only`, `preserve-int-and-bool-flags`

Implement `ex3_coerce_kwargs(kwargs)`. Return a NEW dict where every entry whose value is a Python `float` (NOT int, NOT bool) has been replaced with `t.tensor(float(value))`. Everything else is identity pass-through.

Rules:

- `float` value → `t.tensor(float(v))` (0-D float32 tensor).
- `int` value → pass-through (control-flag use: `dim`, `step`, `groups`).
- `bool` value → pass-through (control-flag use: `keepdim`, `unbiased`).
- `torch.Tensor` value → pass-through unchanged (identity).
- `MiniTensor` value → pass-through (unbox happens later, in a different stage).
- All other types (`tuple`, `None`, `str`, ...) → pass-through.
- Keys preserved exactly. Dict insertion order preserved (Python 3.7+ semantics).

Examples:

```
coerce_kwargs({'min': 0.0, 'max': 1.0, 'keepdim': True})
  → {'min': tensor(0.0), 'max': tensor(1.0), 'keepdim': True}

coerce_kwargs({'dim': 1, 'step': 2})
  → {'dim': 1, 'step': 2}    # ints stay (axis indices)

coerce_kwargs({})
  → {}
```

Why float-only here (and NOT int): kwargs like `dim` and `step` are axis indices — passing a 0-D tensor where the raw torch op expects a Python int raises a `TypeError` deep in the C++ layer. Floats, by contrast, are almost always numeric parameters (`min`, `max`, `eps`, `p` for dropout) and should be tensors so the Recipe stores tensors everywhere.

In [ ]:
def ex3_coerce_kwargs(kwargs):
    out = {}
    for k, v in kwargs.items():
        # bool is a subclass of int; both are NOT coerced.
        # Only Python float gets promoted to a 0-D tensor.
        if isinstance(v, float) and not isinstance(v, bool):
            out[k] = t.tensor(float(v))
        else:
            out[k] = v
    return out


<details><summary>Solution</summary>

```python
def ex3_coerce_kwargs(kwargs):
    out = {}
    for k, v in kwargs.items():
        # bool is a subclass of int; both are NOT coerced.
        # Only Python float gets promoted to a 0-D tensor.
        if isinstance(v, float) and not isinstance(v, bool):
            out[k] = t.tensor(float(v))
        else:
            out[k] = v
    return out
```

**Float-only coercion is the design choice.** Most kwargs that are floats are numeric parameters (`min`, `max`, `eps`, dropout `p`) that the back fn wants as tensors. Ints are almost always axis indices (`dim`, `step`, `groups`) — raw torch ops type-check these as Python ints. Coercing ints would break those ops.

**`isinstance(v, float) and not isinstance(v, bool)`.** Belt-and-suspenders: `bool` is a subclass of `int`, not of `float` in Python — so `isinstance(True, float)` is already False, and the second clause is technically redundant. We keep it for clarity and to match ex1's defensive style. The same belt-and-suspenders appears in PyTorch's own type-check helpers.

**New dict, not in-place mutation.** `coerce_kwargs` is in the wrapper hot path; callers reuse the original `kwargs` dict for Recipe.kwargs storage (the un-coerced version). Mutating in place would corrupt the Recipe.

**This complements ex2's `coerce_args`.** Same numeric promotion, different container. The wrapper calls `coerce_args(args)` and `coerce_kwargs(kwargs)` in sequence before invoking the raw forward fn.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()